# BGPsec, on the Wire

In the [Introduction](Introduction.md) you read how per-hop signature
chains close the path-validation gap that route origin validation (ROV) leaves open. On
the real internet, though, BGPsec's footprint is almost nothing. In this you will
run a simulation of a small internet whose
routers actually speak BGPsec, watch the signatures accumulate hop by hop inside real 
BGP packets, defeat a path forgery with them, and then examine why BGPsec isn't used 
on the internet.

### What is actually running

Six BGP autonomous systems in [Containerlab](Containerlab.md), each
its own container, joined by dedicated point-to-point links:

```
AS64602 ──── AS64601 ──── AS64603 ──── AS64604 ──── AS64605
 origin       transit      transit      transit      observer
                                            │
                                        AS64666
                                        attacker
```

`AS64602` owns `10.64.2.0/24` and announces it; the route travels left to
right. `AS64604` is where you will do most of your watching — it peers with
the attacker, so it is the router that has to judge every claim. `AS64666`
has no special privilege of any kind: same software, its own legitimate
certificate, and nothing but the ability to send a BGP message.

Alongside the routers runs a complete, self-signed RPKI: a CA (**Krill**), a
publication point (**rsyncd**), a validator (**Routinator**), and NIST's
validation engine (**SRx-server**) that five of the routers ask for second
opinions. None of it is connected to the real RPKI.

Each router-to-router link is its own virtual wire. This will let you capture 
a single link and see precisely what one AS hands to the next.

> **Note:** Run this notebook in the repository's GitHub Codespace. The first
> `lab.deploy()` takes 2–3 minutes, and if your Codespace was stopped and
> resumed, re-run `lab.deploy()` before anything else — the virtual wires
> between containers do not survive a stop. The [Containerlab guide](Containerlab.md)
> covers setup, timings, and what to do when something looks stuck, and the
> [Task 2 guidance](Task-2-bgpsec-lab.md) covers what to look for in each step.

---

## Task 0: Setup and Lab Access

`labkit` is the lab's driving library — every cell in this notebook goes
through it. Preflight checks this machine can host the lab and, if it cannot,
names the specific thing to fix.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path("containerlab/lab").resolve()))
import labkit as lab

lab.preflight()

In [ ]:
lab.deploy()

The lab comes up with nothing validated and nothing signed — plain BGP, the
protocol running on trust. BGPsec does not replace that machinery; it rides on
the same RPKI that route origin validation (ROV) uses. So before switching it
on, give the lab
the same origin baseline the real internet has: a ROA authorising AS64602 to
originate its prefix.

In [ ]:
if not lab.vrp_count():
    lab.add_roa()          # 10.64.2.0/24 => AS64602, and waits for the validator

REAL = [64603, 64601, 64602]   # the legitimate path, as AS64604 sees it
lab.status()

---

## Task 1: BGPsec Concepts

Answer these from the [Introduction](Introduction.md). Answer Q1–Q5 **before**
you read its section *Task 1 Background: Why Deployment Is Hard* — they ask you
to predict what that section explains.

### Task 1.1 — Predict the deployment obstacles

Think through what actually changes for a network operator who turns BGPsec on tomorrow.

**Q1** The first adopter. You enable BGPsec and none of your neighbors have. What protection do you actually gain today?

*Your answer for Q1:*

Replace this line with your answer.

**Q2** The unsigned majority. Most of the internet still sends plain BGP updates that you *must* keep accepting. What does an attacker do with that fact?

*Your answer for Q2:*

Replace this line with your answer.

**Q3** The router's workload. A plain BGP update carries an AS_PATH that is a few bytes to copy; a BGPsec update carries one signature per AS hop. What has to happen on a router holding a full internet routing table — roughly a million prefixes — and what happens during a convergence storm?

*Your answer for Q3:*

Replace this line with your answer.

**Q4** The keys. Signing requires private keys on (or behind) every border router in your network, and certificates that other networks fetch to verify you. What ongoing work does that create, and what breaks when it goes wrong?

*Your answer for Q4:*

Replace this line with your answer.

**Q5** Origin validation and BGPsec are both signed, RPKI-rooted mechanisms. What is different about BGPsec that made one of them easy to deploy and the other one hard?

*Your answer for Q5:*

Replace this line with your answer.

### Task 1.2 — Check your understanding

**Q6** In one sentence each, what does a router certificate bind together, and what does the BGPsec_PATH attribute carry beyond an ordinary AS_PATH?

*Your answer for Q6:*

Replace this line with your answer.

**Q7** Why does each AS sign over the ASN it is sending the route to? What attack would be possible if it signed only over the prefix and its own ASN?

*Your answer for Q7:*

Replace this line with your answer.

**Q8** An attacker announces a prefix with a forged path ending in the real owner's ASN. Explain why this passes origin validation but fails BGPsec path validation.

*Your answer for Q8:*

Replace this line with your answer.

**Q9** A route leak is propagated by ASes that all sign correctly. What does BGPsec report, and why is that answer both correct and unhelpful?

*Your answer for Q9:*

Replace this line with your answer.

**Q10** Why does a single non-BGPsec AS in the middle of a path undo the protection for everyone downstream — and why does that make early adoption unattractive?

*Your answer for Q10:*

Replace this line with your answer.

---

## Task 2: Run BGPsec on a Live Network

### Task 2.1 — The identity layer: router keys in the RPKI

A ROA answers "who may originate this prefix". BGPsec needs a second kind of
statement: "this AS signs with this key". Each router in the lab holds a key
pair, and the RPKI publishes a **router certificate** binding the public key
to the AS number.

A router's key is identified by its **SKI** — Subject Key Identifier, a SHA-1
hash of the public key. It is the handle by which a signature says "here is
which key to check me with". Look at the lab's key vault:

In [ ]:
for asn, ski in sorted(lab.skis().items()):
    print(f"AS{asn}  SKI {ski}")

Those are the keys as they exist on disk. Now ask the RTR feed — the same
small protocol that delivers VRPs to routers — what the *validator* is telling
routers about them:

In [ ]:
print(lab.rtr_dump())

Find the `Router Key` PDUs and match one up by hand: pick an AS from the vault
listing and find its SKI in the RTR output. Same twenty bytes, arrived by a
different route — one read off disk, one delivered by the validator after
checking the certificate chain. Before this exercise is over you will see the
same value a third time, inside a packet.

In [ ]:
lab.check("the validator serves a router key for every AS in the lab",
          lab.router_key_count() == 6, f"{lab.router_key_count()} router keys over RTR")
lab.check("and the origin's ROA is being served alongside them",
          lab.vrp_count() == 1, f"{lab.vrp_count()} VRP(s)")
lab.checkpoint("2.1")

### Task 2.2 — Turn BGPsec on

The next cell switches every router into BGPsec mode: each one now **signs**
what it forwards and **verifies** what it receives. The switch restarts the
routing daemons, so give it a minute; the wait polls and returns as soon as
the real route is back.

In [ ]:
lab.set_part("c")
lab.attack("stop")                       # make sure the board is clean
lab.wait_for_best(lab.VICTIM, as_path=REAL)

In [ ]:
lab.show(lab.VICTIM)

Compare this against what a plain-BGP table shows. The route has two
validation results now — **origin**, which is route origin validation (ROV)
against the ROA, and **path**, which is new. Both read `valid`.

The path result is not a lookup. To produce it, AS64604 verified a chain of
signatures: AS64602 signed the announcement to AS64601, AS64601 signed what it
received onward to AS64603, AS64603 onward to AS64604 — and every signature
checked out against a key the RPKI vouches for. Ask the router to show you the
chain it verified, and check each hop's SKI against the vault from Task 2.1:

In [ ]:
vault = lab.skis()

for asn, ski in lab.best(lab.VICTIM).segments:
    match = "matches the RPKI" if vault.get(asn) == ski else "DOES NOT MATCH"
    print(f"  AS{asn}  signed with {ski}  ({match})")

That is the same identity seen from three directions: a key on disk, a Router
Key PDU delivered by the validator, and now a signature inside a routing
message. The agreement of the three is the entire mechanism — it is what lets
AS64604 believe a claim made by AS64602, a network it has never exchanged a
packet with.

In [ ]:
r = lab.best(lab.VICTIM)
lab.check("the real route validates on both axes",
          r.origin_state == "valid" and r.path_state == "valid", str(r))
lab.check("it arrived as a signed BGPsec path, not a plain one", r.signed, str(r))
lab.check("every signature is keyed to the SKI the RPKI publishes",
          all(vault.get(asn) == ski for asn, ski in r.segments) and len(r.segments) == 3,
          ", ".join(f"AS{a}" for a, _ in r.segments))
lab.checkpoint("2.2")

### Task 2.3 — Watch the signatures accumulate

The next cell captures live BGP on each of the four inter-AS links in turn,
forces a re-announcement so there is something to catch, and dissects the
`BGPsec_PATH` attribute out of the packets. It takes a couple of minutes —
four captures, each with a session reset.

In [ ]:
growth = lab.signature_growth()

for label, hops, sigs in growth:
    print(f"{label:22} {hops} hop(s), {sigs} signature(s)")

Notice how the number of signatures increases with the hops. Each
network that forwards the route signs over what it received *and over the AS
it is sending to*, so the block grows by exactly one per hop and the count
always equals the length of the path.

Its important to note that the signature is signed over what is recieved *and the next AS in the hop*.

In [ ]:
counts = [s for _, _, s in growth]
lab.check("the signature block grows by exactly one at each hop",
          counts == [1, 2, 3, 4], f"signatures per link, origin outwards: {counts}")
for label, hops, sigs in growth:
    lab.check(f"{label}: signatures match hop count", hops == sigs)
lab.checkpoint("2.3")

### Task 2.4 — The attack this stops

Remember from the [Introduction](Introduction.md): the attack ROV cannot stop is a **path forgery**.
AS64666 announces the victim's prefix with the path `64666 64602`, claiming "I am
AS64666, I have a direct link to AS64602, and they told me they own this
prefix". The origin in that claim, AS64602 is the *real* owner, so ROV checks the
(prefix, origin) pair against the ROA and finds it authorised. network without BGPsec, 
the forgery is accepted.

AS64666 is about to make exactly that announcement here. Notably, it has a
genuine, CA-issued router certificate of its own. It can sign. Predict, before
you run the cell, whether that helps it.

In [ ]:
lab.attack("forge")

# Wait for the forgery to arrive *and* be judged — a route appears in the
# table a moment before the validator has answered for it.
forged = lab.wait_for_route(lab.VICTIM, via=lab.ATTACKER_PEER, settled=True)

In [ ]:
lab.show(lab.VICTIM)

**Q11** Look at both BGPsec paths. If these were normal BGP paths, which one would routers prefer?

*Your answer for Q11:*

Replace this line with your answer.

**Q12** Notice that the path attribute of the second path, which contains AS64666, is invalid. Use what you know about digital signatures and BGPsec to deduce why.

*Your answer for Q12:*

Replace this line with your answer.

See how there are now two paths, but the second's path is invalid.

The **origin** axis still says `valid`. The ROA authorises AS64602 to
originate this prefix; the forged path names AS64602 as origin; those agree.
ROV is fooled here exactly as it would be on the real internet.

The **path** axis says `invalid`. To produce a verifiable `64666 64602`,
AS64666 needed AS64602's signature over that path — a signature naming AS64666
as the recipient. AS64602 never made one, AS64666 cannot make it without
AS64602's private key, and its own perfectly genuine key is vouched for as
AS64666, not AS64602. The chain fails at the fabricated link.

In [ ]:
lab.check("the forged path is rated Invalid on the path axis",
          forged.path_state == "invalid", str(forged))
lab.check("ROV is still fooled: the origin axis reads valid",
          forged.origin_state == "valid", str(forged))
lab.check("the route is dropped by policy", forged.ignored, str(forged))
lab.check("traffic follows the real, fully signed path",
          lab.best(lab.VICTIM).as_path == REAL)
lab.checkpoint("2.4")

In [ ]:
lab.attack("stop")
lab.wait_for_gone(lab.VICTIM, via=lab.ATTACKER_PEER)

### Task 2.5 — The trust is revocable

The signatures verify because the RPKI vouches for the public keys behind
them. That vouching is a published statement, and statements can be withdrawn.

Revoke AS64602's router certificate at the CA, and watch what does **not**
happen: no router is reconfigured, no session resets, and AS64602 keeps its
private key and carries on signing exactly as before. The only thing that
changes is a certificate published by a third party.

In [ ]:
lab.revoke_router_key(64602)

In [ ]:
lab.show(lab.VICTIM)

The route is **gone** — not de-preferred, not flagged. Since this configuration drops
invalid routes, and there is no certificate for AS64604, the route is not propogated.
AS64602 has not noticed: it is still announcing, still signing, still
sending the same bytes down the same wire. Everything that changed happened at
a certificate authority and travelled to the routers as data.

This means that whoever operates the CA has the power to remove live routes from
the internet on a whim.

Put it back before moving on:

In [ ]:
after = lab.routes(lab.VICTIM)
lab.check("no path to the prefix verifies any more",
          not any(r.path_state == "valid" for r in after),
          "; ".join(str(r) for r in after) or "no path to the prefix at all")

lab.restore_router_key(64602)
lab.wait_for_best(lab.VICTIM, as_path=REAL)
lab.checkpoint("2.5")

### Task 2.6 — What happens when one network has not deployed it

Everything so far assumed all five networks on the path speak BGPsec. On the real
internet that assumption fails badly: the global RPKI holds only a handful of BGPsec router certificates (2 in October 2026), against about a million announced routes.

The next cell switches the lab into **partial
deployment**: the origin, AS64602, goes back to plain BGP with no keys and no
signatures while everyone else keeps BGPsec on. Because dropping every
unverifiable route would now mean disconnecting from the unsigned majority,
the routers' policy weakens from *drop* to *de-prefer*.

In [ ]:
lab.set_part("d")
lab.attack("stop")
lab.wait_for_best(lab.VICTIM, as_path=REAL)
lab.wait_for_route(lab.OBSERVER)       # the far end hears the route a moment later

In [ ]:
lab.show(lab.VICTIM)
lab.show(lab.OBSERVER)

The route still arrives, as a network that has not deployed BGPsec is not cut
off. However, the origin never signed, so the
chain was never started, and no downstream router can verify anything: path
`invalid` at AS64604, and at the far end AS64605 can still prove *who owns the
prefix*, but nothing at all about the path.

Four networks paid the full cost of deployment here. Because one did not, all
four got nothing for it.

In [ ]:
r = lab.best(lab.VICTIM)
lab.check("the route still arrives — a non-BGPsec network is not cut off",
          r.as_path == REAL, str(r))
lab.check("but the path can no longer be verified",
          r.path_state == "invalid", str(r))
lab.check("and it is de-preferred rather than dropped",
          r.best and r.localpref <= r.localpref_base, str(r))
obs = lab.best(lab.OBSERVER)
lab.check("at the far end: origin provable, path not signed at all",
          obs.origin_state == "valid" and not obs.signed, str(obs))
lab.checkpoint("2.6")

---

### Task 2.7 — Reflection

Answer from what you observed, citing the task step where you observed it.

**Q13** *What a signature commits to.* In Task 2.3 the signature block grew by one per hop, and each signature covered the announcement *and the AS it was being sent to*. Why must the recipient be inside the signature? What could an AS do to defeat path validation if an AS only signed over the received hops?

*Your answer for Q13:*

Replace this line with your answer.

**Q14** *Two validators, two verdicts.* In Task 2.4 the same forged announcement was rated **valid** on the origin axis and **invalid** on the path axis. Is that correct? Explain where each rating comes from and what it means.

*Your answer for Q14:*

Replace this line with your answer.

**Q15** *Where the power went.* In Task 2.5 a route vanished from AS64604's table without any router being touched. Who, exactly, gained what power when this lab turned BGPsec on? Explain one way that power is an operational necessity and one way it is a new risk.

*Your answer for Q15:*

Replace this line with your answer.

**Q16** *The missing footprint.* Put Task 2.6 next to the real-world numbers (about 400,000 ROAs, about a million VRPs, against a handful of BGPsec router certificates: 2 in October 2026). Using only things you observed in this exercise, give the two strongest reasons deployment looks like that. Then explain why an attacker on a partially deployed internet does not even need to forge signatures.

*Your answer for Q16:*

Replace this line with your answer.

---

## Finishing up

Reset puts the lab back to its starting state; destroy tears the containers
down entirely. In a Codespace, stopping the machine achieves the same thing.
If you want to resume, rember to run lab.deploy() again.

In [ ]:
lab.reset()
# lab.destroy()